# Full Load Fact

This notebook keeps the original execution flow and variable names. The refactor is intentionally limited to cleanup, documentation, configurable existing parameters, and correction of the file-move variable typo.

## Execution outline
1. Load the existing Databricks utility setup.
2. Read configurable catalog, source, storage, table, and metadata parameters.
3. Read CSV files from landing and add file metadata.
4. Append raw records to Bronze.
5. Move processed files from landing to processed.
6. Parse and standardize dates, customer IDs, quantities, and remove duplicate records for Silver.
7. Join product information.
8. Create or merge the Silver Delta table.
9. Rename and select Gold columns.
10. Aggregate sales by month, product, and customer.
11. Merge the resulting Gold data into the parent fact table.
12. Validate the resulting datasets and review performance notes.


In [0]:
%run ../1_setup/utilities


## 1. Environment setup and dependency installation

Databricks provides Spark, Delta Lake, and `dbutils` in the cluster runtime. The existing `%run` utility is retained because it is part of the original notebook execution flow. No additional package installation is introduced.


## 2. Configuration management

The original configuration variable names are retained. Their values are exposed as Databricks widgets so the same notebook can be run for another catalog, source, storage location, or target table without changing the code.


In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data source", "fact_orders")
dbutils.widgets.text("STORAGE_ACCOUNT", "fmcgaccount.dfs.core.windows.net")
dbutils.widgets.text("CONTAINER_NAME", "sports-bar-dp")
dbutils.widgets.text("PARENT_ORDERS_FACT_TABLE", "fmcg.gold.fact_orders")
dbutils.widgets.text("DEFAULT_MARKET", "India")
dbutils.widgets.text("DEFAULT_PLATFORM", "Sports bar")
dbutils.widgets.text("DEFAULT_CHANNEL", "Aquisition")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data source")
STORAGE_ACCOUNT = dbutils.widgets.get("STORAGE_ACCOUNT")
CONTAINER_NAME = dbutils.widgets.get("CONTAINER_NAME")
PARENT_ORDERS_FACT_TABLE = dbutils.widgets.get("PARENT_ORDERS_FACT_TABLE")
DEFAULT_MARKET = dbutils.widgets.get("DEFAULT_MARKET")
DEFAULT_PLATFORM = dbutils.widgets.get("DEFAULT_PLATFORM")
DEFAULT_CHANNEL = dbutils.widgets.get("DEFAULT_CHANNEL")

landing_path = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT}/{data_source}/landing/"
processed_path = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT}/{data_source}/processed/"
bronze_table_name = f"{catalog}.bronze.{data_source}"
silver_table_name = f"{catalog}.silver.{data_source}"
gold_table_name = f"{catalog}.gold.sb_fact_{data_source}"
prdouct_table_name = f"{catalog}.silver.products"

print(f"Catalog: {catalog}")
print(f"Data Source: {data_source}")
print(f"Landing Path: {landing_path}")
print(f"Processed Path: {processed_path}")


## 3. Utility functions

No new utility functions are introduced. The notebook continues to use the existing Databricks/Spark APIs and the shared utility notebook loaded above.


## 4. Core algorithm implementation

The following cells retain the original Bronze → Silver → Gold → parent fact execution flow. Transformations and merge keys are kept unchanged.


In [0]:
df = spark.read.option("header", True).csv(f"{landing_path}/*.csv").withColumn("read_timestamp", F.current_timestamp()).select("*", "_metadata.file_name", "_metadata.file_size")

try:
    display(df)
except Exception as e:
    dbutils.notebook.exit(f"No files found for {data_source}")


In [0]:
(
    df.write
    .format("delta")
    .mode("append")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(bronze_table_name)
)


In [0]:
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(file_info.path, f"{processed_path}/{file_info.name}", True)


In [0]:
import pyspark.sql.functions as F

spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

clean_date = F.trim(
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

date_formats = [
    "dd-MM-yyyy", "d-M-yyyy",
    "MM-dd-yyyy", "M-d-yyyy",
    "MMMM dd, yyyy", "MMMM d, yyyy", "MMM dd, yyyy", "MMM d, yyyy",
    "yyyy-MM-dd", "yyyy-M-d",
    "MM/dd/yyyy", "M/d/yyyy",
    "dd/MM/yyyy", "d/M/yyyy",
    "yyyy/MM/dd", "dd-MMM-yyyy",
    "yyyy-MM-dd HH:mm:ss", "MM/dd/yyyy HH:mm:ss"
]

parsed_date = F.coalesce(*[
    F.to_date(F.try_to_timestamp(clean_date, F.lit(fmt)))
    for fmt in date_formats
])

silver_df = (
    spark.read.table(bronze_table_name)
    .filter(F.col("order_qty").isNotNull())
    .withColumn(
        "customer_id",
        F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
        .otherwise("9999")
        .cast("string")
    )
    .withColumn("order_placement_date", F.to_date(parsed_date, "MMMM dd, yyyy"))
    .dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])
    .withColumn("order_qty", F.col("order_qty").cast("float").cast("int"))
)

display(silver_df)


In [0]:
prdouct_table = spark.table(prdouct_table_name)
silver_df = silver_df.join(prdouct_table, on="product_id", how="inner").select(silver_df["*"], prdouct_table["product_code"])
print(silver_df.columns)


In [0]:
if not spark.catalog.tableExists(silver_table_name):
    (
        silver_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .option("delta.enableChangeDataFeed", "true")
        .saveAsTable(silver_table_name)
    )
else:
    silver_prev_table = DeltaTable.forName(spark, silver_table_name)
    (
        silver_prev_table.alias("prev")
        .merge(
            silver_df.alias("new"),
            "prev.order_placement_date = new.order_placement_date AND prev.customer_id = new.customer_id AND prev.product_id = new.product_id AND prev.order_id = new.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )


In [0]:
gold_table_renming_map = {
    "order_id": "order_id",
    "order_placement_date": "date",
    "customer_id": "customer_code",
    "product_code": "product_code",
    "order_qty": "sold_quantity",
    "product_code": "product_code",
}
gold_df = spark.read.table(silver_table_name).withColumnsRenamed(gold_table_renming_map).select("date", "product_code", "customer_code", "sold_quantity","order_id")


In [0]:
if not spark.catalog.tableExists(gold_table_name):
    print("creating New Table")
    (
        gold_df.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .option("tblproperties.delta.enableChangeDataFeed", "true")
        .saveAsTable(gold_table_name)
    )
else:
    print("Merging into existing Table")
    column_mapping = {col: col for col in gold_df.columns}
    merge_child_to_parent_dim(
        spark=spark,
        child_table=gold_df,
        parent_table_name=gold_table_name,
        column_mapping=column_mapping,
        merge_key=["date", "order_id", "product_code", "customer_code"]
    )

In [0]:
gold_df = (
    gold_df
    .withColumn("month_start", F.trunc("date", "MM"))
    .groupby("month_start", "product_code", "customer_code")
    .agg(F.sum("sold_quantity").alias("sold_quantity"))
    .withColumnRenamed("month_start", "date")
)

display(gold_df)


In [0]:
customer_column_mapping = {
    "date": "date",
    "product_code": "product_code",
    "customer_code": "customer_code",
    "sold_quantity": "sold_quantity"
}

merge_child_to_parent_dim(
    spark=spark,
    child_table=gold_df,
    parent_table_name=PARENT_ORDERS_FACT_TABLE,
    column_mapping=customer_column_mapping,
    merge_key=["customer_code", "product_code", "date"]
)
